## Starting Notebook for Assignment Part 1 A

The assignment will consider a new type of puzzle and use it as a test case for developing the following skills:

* Creating your own puzzles of different levels of difficulty.
* Testing different search algorithms and parameters, as provided by the `bbqsearch` Python package.
* Devising heuristics to guide the search.
* Carrying out systematic comparisons between different search algorithms on a variety of different problems and using different heuristics.

If you have done the search exercises you should be familiar with how this software works. However, if you need some revision of how to run the search algorithm and the various options available, you should go back to the first BBQ Search exercise notebook (`BBQ_Search_A_Intro_Jug_Pouring.ipynb`), which explains the basic usage. You may also want to revisit some of the examples in the other exercises. For this part of the assignment, the exercises on the **8-Puzzle** and **Sliding Blocks Puzzles** are of particular relevance.


# Traffic Jam Puzzles

There is a variety of sliding block puzzle (see `BBQ_Search_F_Sliding_Blocks.ipynb`) that we shall call a  _**Traffic Jam Puzzle**_, exemplified by a commercial toy product that is often sold under the name _**Rush Hour**_.

<center>
<img src="https://s3.eu-west-2.amazonaws.com/bb-ai.net/graphics/rush-hour-game.jpg"  width="200">
</center>   

The main difference between these problems and the general type of sliding block problem that we implemented in the earlier exercise is that the possible movements of the moving objects (vehicles) are constrained to be either in the vertical or in the horizontal direction.



## Setup

As usual we need to download and import from Brandon's `bbqsearch` package:

In [ ]:
%pip install bbqsearch
from bbqsearch import SearchProblem, search

### The `TrafficJam` Problem Class
The problem you will experiment on has been created for you by the following class definitions. `TrafficState` defines the states within the problem search space, which specify locations and movement possibilities of vehicles, obstacles and traffic cones. It also provides a `display` method that will show a graphic representation of the state.

The main problem class is `TrafficJam`. You can create an instance of this by supplying initial and goal states, using a command such as `jam_prob = TrafficJam( tj_initial, tj_goal)`.

**You should not modify or extend the `TrafficJam` or `TrafficState` classes.** You may wish to examine them to see how the problem was coded within the `bbqsearch` framework, but this is not necessary. The code is very similar to that used for the Sliding Blocks puzzle implementation but with additional constraint handling. Most of the code is actually handling the graphics for the `display` method.

But you will need to understand the format of the state representation specification data-structure that is used to create particular `TrafficState` objects corresponding to initial states and goal states. This is a list of lists of symbols representing the contents of each cell in a grid.
This state specification format is explained in the cells following the class specifications.

#### Code of `TrafficState` and `TrafficJam` Classes

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from copy import deepcopy

plt.ioff()

COLORS = ["lightgray", "red", "blue", "green", "yellow",
          "orange", "purple", "pink", "brown"]

class TrafficState:

    neighbour_offset = {"left":(0,-1), "right":(0,+1),
                        "down":(+1,0), "up":(-1,0)}
    directions = {"-":["left","right"], "|":["up","down"]}
    cone_directions = ["left","right","up","down"]

    def __init__(self, blockstate, colors=COLORS):
        self.blockstate = blockstate
        self.nrows = len(blockstate)
        self.ncols = len(blockstate[0])
        self.blocknums = {int(x[1:]) for row in blockstate for x in row
                          if isinstance(x,str) and x not in {"#","*"}}
        self.blocknumlist = sorted(self.blocknums)
        self.colors = colors

    def __repr__(self):
        return str(self.blockstate)

    @staticmethod
    def blocknum(x):
        return int(x[1:]) if isinstance(x,str) and x not in {"#","*"} else None

    def blockcells(self, blocknum):
        return [(r,c) for r in range(self.nrows) for c in range(self.ncols)
                if self.blocknum(self.blockstate[r][c]) == blocknum]

    def movement_type(self, blocknum):
        symbols = {self.blockstate[r][c][0] for r,c in self.blockcells(blocknum)}
        if symbols <= set("<=>"): return "-"
        if symbols <= set("^|v"): return "|"
        raise ValueError(f"Inconsistent movement symbols for object {blocknum}")

    def free_cell(self, direction, cell):
        r,c = cell
        dr,dc = self.neighbour_offset[direction]
        nr,nc = r+dr,c+dc
        if not (0 <= nr < self.nrows and 0 <= nc < self.ncols): return False
        neigh = self.blockstate[nr][nc]
        b = self.blocknum(self.blockstate[r][c])
        return neigh == 0 or (b is not None and self.blocknum(neigh) == b)

    def free_block(self, direction, blocknum):
        return all(self.free_cell(direction,c) for c in self.blockcells(blocknum))

    def possible_moves(self):
        moves = []
        for b in self.blocknumlist:
            for d in self.directions[self.movement_type(b)]:
                if self.free_block(d,b):
                    moves.append((b,d))

        for r in range(self.nrows):
            for c in range(self.ncols):
                if self.blockstate[r][c] == "*":
                    for d in self.cone_directions:
                        if self.free_cell(d,(r,c)):
                            moves.append(("*",(r,c),d))
        return moves

    def next_state(self, move):
        nxt = deepcopy(self.blockstate)

        if move[0] == "*":
            _,(r,c),d = move
            dr,dc = self.neighbour_offset[d]
            nxt[r][c] = 0
            nxt[r+dr][c+dc] = "*"
        else:
            b,d = move
            cells = self.blockcells(b)
            vals = [self.blockstate[r][c] for r,c in cells]
            for r,c in cells:
                nxt[r][c] = 0
            dr,dc = self.neighbour_offset[d]
            for (r,c),v in zip(cells,vals):
                nxt[r+dr][c+dc] = v

        return TrafficState(nxt,self.colors)

    def blockcolor(self, b):
        return self.colors[1+(b-1)%(len(self.colors)-1)]

    def color_key(self):
        return {b:self.blockcolor(b) for b in self.blocknumlist}

    def figure(self, scale=0.6):
        fig,ax = plt.subplots(figsize=(self.ncols*scale,self.nrows*scale))
        plt.close(fig)
        ax.set_xlim(0,self.ncols)
        ax.set_ylim(0,self.nrows)
        ax.set_aspect("equal")
        ax.set_axis_off()

        for r in range(self.nrows):
            for c in range(self.ncols):
                y = self.nrows-r-1
                celltype = self.blockstate[r][c]

                ax.add_patch(patches.Rectangle(
                    (c,y),1,1,linewidth=1,edgecolor="gray",
                    facecolor="lightgray"))

                if celltype == "#":
                    ax.add_patch(patches.Rectangle(
                        (c+.08,y+.08),.84,.84,linewidth=2.6,
                        edgecolor="black",facecolor="#A0522D"))

                elif celltype == "*":
                    base = .35
                    ax.add_patch(patches.Polygon(
                        [(c+.32,y+base),(c+.68,y+base),(c+.54,y+base+.48)],
                        closed=True,linewidth=1.1,edgecolor="black",
                        facecolor="orangered"))
                    ax.add_patch(patches.Polygon(
                        [(c+.40,y+base+.14),(c+.63,y+base+.14),
                         (c+.59,y+base+.26),(c+.44,y+base+.26)],
                        closed=True,linewidth=0,facecolor="white"))
                    ax.add_patch(patches.Rectangle(
                        (c+.28,y+base-.08),.44,.07,linewidth=1,
                        edgecolor="black",facecolor="firebrick"))

        for b in self.blocknumlist:
            cells = self.blockcells(b)
            rows = [r for r,c in cells]
            cols = [c for r,c in cells]
            r1,r2,c1,c2 = min(rows),max(rows),min(cols),max(cols)
            x,y = c1,self.nrows-r2-1
            w,h = c2-c1+1,r2-r1+1
            col = self.blockcolor(b)

            ax.add_patch(patches.FancyBboxPatch(
                (x+.16,y+.16),w-.30,h-.30,
                boxstyle="round,pad=0.02,rounding_size=0.18",
                linewidth=1.5,edgecolor="black",facecolor=col))

            if w > h:
                ax.add_patch(patches.Rectangle(
                    (x+w*.40,y+.24),w*.20,h-.48,
                    linewidth=1,edgecolor="black",facecolor="white"))
            elif h > w:
                ax.add_patch(patches.Rectangle(
                    (x+.24,y+h*.40),w-.48,h*.20,
                    linewidth=1,edgecolor="black",facecolor="white"))

            ax.text(x+w/2,y+h/2,str(b),ha="center",va="center",
                    fontsize=11,weight="bold")

        ax.add_patch(patches.Rectangle(
            (0,0),self.ncols,self.nrows,linewidth=3,
            edgecolor="black",facecolor="none"))

        return fig

    def display(self):
        display(self.figure())


class TrafficJam(SearchProblem):

    def __init__(self, initial_state, goal, colors=COLORS):
        self.initial_state = TrafficState(initial_state,colors)
        self.goal = TrafficState(goal,colors)
        self.colors = colors

    def info(self):
        print("Solve the following traffic jam problem.")
        print("Get from this initial state:")
        self.initial_state.display()
        print("To a state incorporating the following object position(s):")
        self.goal.display()

    def possible_actions(self, state):
        return state.possible_moves()

    def successor(self, state, action):
        return state.next_state(action)

    def goal_test(self, state):
        for r in range(state.nrows):
            for c in range(state.ncols):
                g = self.goal.blockstate[r][c]
                if g != 0 and g != state.blockstate[r][c]:
                    return False
        return True

    def cost(self, path, state):
        return len(path)

    def display_action(self, action):
        if action[0] == "*":
            print(("traffic cone",action[1],action[2]))
        else:
            print((self.initial_state.blockcolor(action[0]),action[1]))

    def display_state(self, state):
        state.display()

    def display_state_path(self, actions):
        s = self.initial_state
        self.display_state(s)
        for a in actions:
            self.display_action(a)
            s = self.successor(s,a)
            self.display_state(s)

## Specifying Traffic Jam states

A Traffic Jam state is specified as a rectangular list of rows, with one value for each grid cell.

- `0` represents an empty cell.
- Horizontal vehicles are represented by `"<n"`, `"=n"` and `">n"` for the left end, middle and right end of vehicle `n`.
- Vertical vehicles are represented by `"^n"`, `"|n"` and `"vn"` for the top end, middle and bottom end of vehicle `n`.
- The number `n` identifies the vehicle and should be unique to that vehicle. Vehicles may have different lengths; longer vehicles simply have more middle cells.
- `"*"` represents a traffic cone. A cone occupies one cell and may move in any orthogonal direction.
- `"#"` represents a fixed obstacle, which cannot move.

The specification is **not fully checked for consistency with this intended interpretation**, so you must ensure that it makes sense. In particular, the cells belonging to each vehicle should form one contiguous horizontal or vertical line, with appropriate end and middle symbols.

A goal state is normally only a **partial specification**. Every non-empty cell in the goal must have the same contents in a state for the goal to be satisfied, but `0` cells in the goal impose no condition. Thus, for example, a goal may specify only the required final position of one vehicle. The goal grid must have the same dimensions as the initial-state grid.

The example below defines an initial state and a goal state, creates `TrafficState` objects from them, and displays both states.

In [ ]:
initial = [
    [0,    "<5", "=5", ">5", 0,    0,   0],
    [0,    0,    "^2", 0,    "*",  "#", 0],
    [0,    0,    "|2", 0,    "^3", "#", 0],
    ["<1", ">1", "v2", 0,    "v3", 0,   0],
    [0,    0,    0,    "<4", ">4", 0,   0],
]

goal = [
    [0,0,0,0,0,0,0],
    [0,0,0,0,0,0,0],
    [0,0,0,0,0,0,0],
    [0,0,0,0,0,"<1",">1"],
    [0,0,0,0,0,0,0],
]

s = TrafficState(initial)
g = TrafficState(goal)
s.display()
g.display()

### Solving the example puzzle

We can now create a `TrafficJam` search problem from the initial and goal states and solve it using the `bbqsearch` search function. The example below uses breadth-first search with loop checking and displays the sequence of states in the solution path.



In [ ]:
test_puzzle = TrafficJam(initial,goal)
test_search = search( test_puzzle, 'BF/FIFO', 100000,
                      loop_check=True, randomise=False,
                      show_state_path=True, return_info=True)

# Over to you!

You can use this example as a starting point for experimenting with other search algorithms, options and heuristics. If you are doing this as part of an assignment, be sure to check the instructions and suggestions in the assignment specification document.

**Now it’s your turn to take the wheel — experiment with the search methods and see what gets through the jam.**